# SHERA ML Data Access Quickstart

Use this notebook to inspect a prepared SHERA ML dataset, read sharded images, select controlled science/nuisance examples, build a difference image, and sample a few ordered pairs. It does not train a model.

## 1. Configure a prepared dataset

Set `PREPARED_ROOT` to a prepared dataset directory containing `manifest.json`, `vector_spaces.json`, `index.jsonl`, `array_shards_manifest.json`, and `shards/`. You can also set the `SHERA_PREPARED_ROOT` environment variable before launching Jupyter.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np

from notebook_setup import setup_paths

REPO_ROOT = setup_paths()

env_root = os.environ.get("SHERA_PREPARED_ROOT")
PREPARED_ROOT = Path(env_root) if env_root else None
# PREPARED_ROOT = Path("/path/to/prepared_dataset")

print("Prepared root:", PREPARED_ROOT if PREPARED_ROOT is not None else "not set")

## 2. Load the catalog

`SampleCatalog` keeps compact metadata arrays and group identities. It does not load image shards into memory.

In [ ]:
from dluxshera.ml import load_sample_catalog

if PREPARED_ROOT is None:
    raise RuntimeError(
        "Set PREPARED_ROOT above, or launch Jupyter with SHERA_PREPARED_ROOT "
        "pointing to a prepared dataset directory."
    )
if not PREPARED_ROOT.exists():
    raise FileNotFoundError(f"Prepared dataset directory does not exist: {PREPARED_ROOT}")

catalog = load_sample_catalog(PREPARED_ROOT)
print(json.dumps(catalog.summary(), indent=2))

## 3. Inspect sample identities

Science groups identify physical perturbation states. Nuisance groups identify registration/rendering nuisance realizations.

In [ ]:
print("science groups:", catalog.science_group_count)
print("nuisance groups:", catalog.nuisance_group_count)
print("science labels:", catalog.parameter_labels[:8])
print("nuisance labels:", catalog.nuisance_labels)
print(json.dumps(catalog.sample_metadata(0), indent=2)[:1200])

## 4. Read one image from the sharded store

In [ ]:
row = 0
array_index = int(catalog.array_indices[row])

with catalog.image_reader(cache_size=4) as reader:
    image = reader[array_index]

print("image shape:", image.shape)
print("image dtype:", image.dtype)
plt.figure(figsize=(4, 4))
plt.imshow(image, origin="lower", cmap="viridis")
plt.colorbar(label="intensity")
plt.title(str(catalog.sample_ids[row]))
plt.show()

## 5. Select controlled science/nuisance examples

In [ ]:
science_id = catalog.science_groups[0]
nuisance_id = catalog.nuisance_groups[0]

same_science_rows = catalog.indices_for_groups(science_groups=[science_id])
same_nuisance_rows = catalog.indices_for_groups(nuisance_groups=[nuisance_id])

print("rows with one science state across nuisance realizations:", same_science_rows[:10])
print("rows with one nuisance realization across science states:", same_nuisance_rows[:10])

## 6. Build and display a difference image

This example fixes the nuisance group and chooses two different science states.

In [ ]:
row_a = int(same_nuisance_rows[0])
row_b = next(
    int(row)
    for row in same_nuisance_rows[1:]
    if catalog.science_group_ids[row] != catalog.science_group_ids[row_a]
)

with catalog.image_reader(cache_size=4) as reader:
    image_a = reader[int(catalog.array_indices[row_a])]
    image_b = reader[int(catalog.array_indices[row_b])]

difference = image_b - image_a
target_delta_z = catalog.fisher_scaled_deltas[row_b] - catalog.fisher_scaled_deltas[row_a]
print("target_delta_z:", target_delta_z)

diff_limit = float(np.nanmax(np.abs(difference)))
if diff_limit == 0.0:
    diff_limit = 1.0

fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
panels = [
    (image_a, "image A", "viridis", None, None),
    (image_b, "image B", "viridis", None, None),
    (difference, "B - A", "coolwarm", -diff_limit, diff_limit),
]
for ax, data, title, cmap, vmin, vmax in panels:
    im = ax.imshow(data, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.show()

## 7. Sample controlled pairs

`PairSampler` draws controlled ordered comparisons without materializing every possible pair.

In [ ]:
from dluxshera.ml import PairPolicy, PairSampler, generate_split_registry

split_registry = generate_split_registry(
    catalog,
    seed=11,
    science_fractions={"train": 1.0},
    nuisance_fractions={"train": 1.0},
)
policy = PairPolicy(
    family_weights={"same_nuisance_different_science": 1.0},
    min_fisher_distance=0.0,
)
sampler = PairSampler(catalog, split_registry, policy)
record = sampler.sample_pair(np.random.default_rng(0))

print(json.dumps(record.to_dict(), indent=2)[:1600])

## 8. Optional split inspection

For real validation/test work, use nontrivial science and nuisance fractions and persist the registry with `write_split_registry`.

In [ ]:
split_registry = generate_split_registry(
    catalog,
    seed=11,
    science_fractions={"train": 0.8, "validation": 0.1, "test": 0.1},
    nuisance_fractions={"train": 0.8, "validation": 0.1, "test": 0.1},
    require_nonempty_nuisance_partitions=catalog.nuisance_group_count >= 3,
)
print(json.dumps(split_registry.counts, indent=2))
print("train science groups:", len(split_registry.science_groups("train")))
print("validation nuisance groups:", len(split_registry.nuisance_groups("validation")))